# Customer Segmentation (RFM) — End-to-End Portfolio Project
**Dataset:** Online Retail II UCI (`/kaggle/input/datasets/nofaamanda/online-retail-ii-uci`)  
**Tools:** Python, SQLite (CTE Queries), K-Means Clustering, Excel Automation

### Business Problems
1. Berapa persen customer yang menyumbang mayoritas revenue? (Pareto / Revenue Concentration)
2. Segmen mana yang berisiko churn dan layak diprioritaskan untuk retensi?

## 0. Setup Library & Deteksi Path Dataset Kaggle

In [ ]:
import os
import glob
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')

# Buat folder kerja di /kaggle/working
os.makedirs('data/clean', exist_ok=True)
os.makedirs('data/sql_exports', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# Deteksi otomatis file di dalam folder dataset Kaggle
INPUT_DIR = '/kaggle/input/datasets/nofaamanda/online-retail-ii-uci'
if os.path.exists(INPUT_DIR):
    files = [os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR) if f.endswith(('.csv', '.xlsx', '.xls'))]
    FILE_PATH = files[0] if files else None
else:
    alt_files = glob.glob('/kaggle/input/**/*retail*.*', recursive=True)
    FILE_PATH = alt_files[0] if alt_files else 'online_retail_II.csv'

print('File dataset yang digunakan:', FILE_PATH)

## TAHAP 1 — Ambil & Bersihkan Data (Python)
- Menggabungkan sheet `Year 2009-2010` dan `Year 2010-2011` jika terpisah.
- Mengecek `df.columns` terlebih dahulu, missing values, dan duplikat.
- Menghapus `Customer ID` kosong, membuang invoice batal (awalan `C`), membuang `Quantity <= 0` atau `Price <= 0`.
- Merapikan tipe data, membuat kolom `Revenue`, dan menyimpan ke `data/clean/retail_clean.csv`.

In [ ]:
# 1. Load Data (Support .xlsx 2 sheet maupun .csv gabungan)
if FILE_PATH.endswith(('.xlsx', '.xls')):
    sheets = pd.read_excel(FILE_PATH, sheet_name=None)
    print('Sheet ditemukan:', list(sheets.keys()))
    df_raw = pd.concat(sheets.values(), ignore_index=True)
else:
    df_raw = pd.read_csv(FILE_PATH)

# 2. Cek ukuran awal, df.columns, dan missing values
print(f'Total baris awal: {len(df_raw):,}')
print('Nama kolom asli (df.columns):', df_raw.columns.tolist())
print('\nMissing values awal per kolom:')
print(df_raw.isnull().sum())

# 3. Pembersihan Bertahap & Tracking Baris Terbuang
df = df_raw.copy()
df.columns = df.columns.str.strip()
df = df.rename(columns={
    'InvoiceNo': 'Invoice',
    'Customer ID': 'CustomerID',
    'UnitPrice': 'Price'
})

audit_steps = []
n0 = len(df)
audit_steps.append(('0. Data Awal (Mentah)', n0, 0))

# Langkah 1: Hapus duplikat identik
df = df.drop_duplicates()
n1 = len(df)
audit_steps.append(('1. Hapus Duplikat Identik', n1, n0 - n1))

# Langkah 2: Hapus Customer ID kosong (NaN)
df = df.dropna(subset=['CustomerID'])
n2 = len(df)
audit_steps.append(('2. Hapus Customer ID Kosong (NaN)', n2, n1 - n2))

# Langkah 3: Buang invoice batal (awalan 'C')
df['Invoice'] = df['Invoice'].astype(str).str.strip()
df = df[~df['Invoice'].str.upper().str.startswith('C')]
n3 = len(df)
audit_steps.append(("3. Buang Invoice Batal (Awalan 'C')", n3, n2 - n3))

# Langkah 4: Buang Quantity <= 0 atau Price <= 0
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
n4 = len(df)
audit_steps.append(('4. Buang Quantity <= 0 atau Price <= 0', n4, n3 - n4))

# Langkah 5: Rapikan tipe data & Buat kolom Revenue
df['CustomerID'] = df['CustomerID'].astype(float).astype(int).astype(str)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Quantity'] = df['Quantity'].astype(int)
df['Price'] = df['Price'].astype(float)
df['Revenue'] = (df['Quantity'] * df['Price']).round(2)

df = df[['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country', 'Revenue']]

audit_df = pd.DataFrame(audit_steps, columns=['Tahap Pembersihan', 'Sisa Baris', 'Baris Terbuang'])
audit_df['% Terbuang dari Awal'] = (audit_df['Baris Terbuang'] / n0 * 100).round(2)

print('\n=== REKAP BARIS TERBUANG PER LANGKAH ===')
display(audit_df)
print(f'Total baris terbuang : {n0 - n4:,} ({(n0 - n4)/n0*100:.2f}%) | Sisa baris bersih : {n4:,} ({n4/n0*100:.2f}%)')

clean_csv_path = 'data/clean/retail_clean.csv'
df.to_csv(clean_csv_path, index=False)
audit_df.to_csv('reports/cleaning_audit_log.csv', index=False)
print(f'Data bersih berhasil disimpan ke: {clean_csv_path}')

## TAHAP 2 — SQL (SQLite + CTE Queries)
- Memuat `data/clean/retail_clean.csv` ke SQLite (`data/retail.db`).
- Query CTE 1: Menghitung **Recency, Frequency, Monetary** per customer.
- Query CTE 2: Menghitung **Revenue per Negara**.
- Query CTE 3: Menghitung **Tren Customer Baru vs Repeat per Bulan**.

In [ ]:
db_path = 'data/retail.db'
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
df.to_sql('transactions', conn, if_exists='replace', index=False)

conn.execute('CREATE INDEX idx_cust ON transactions(CustomerID);')
conn.execute('CREATE INDEX idx_date ON transactions(InvoiceDate);')
conn.execute('CREATE INDEX idx_country ON transactions(Country);')

# QUERY 1: CTE Recency, Frequency, Monetary per Customer
query_rfm = '''
WITH ref_date AS (
    SELECT DATE(MAX(InvoiceDate), '+1 day') AS snapshot_date
    FROM transactions
),
cust_country AS (
    SELECT
        CustomerID,
        Country,
        ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY COUNT(*) DESC) AS rn
    FROM transactions
    GROUP BY CustomerID, Country
),
rfm_calc AS (
    SELECT
        t.CustomerID,
        CAST(JULIANDAY((SELECT snapshot_date FROM ref_date)) - JULIANDAY(MAX(DATE(t.InvoiceDate))) AS INTEGER) AS Recency,
        COUNT(DISTINCT t.Invoice) AS Frequency,
        ROUND(SUM(t.Revenue), 2) AS Monetary
    FROM transactions t
    GROUP BY t.CustomerID
)
SELECT
    r.CustomerID,
    c.Country,
    r.Recency,
    r.Frequency,
    r.Monetary
FROM rfm_calc r
LEFT JOIN cust_country c
    ON r.CustomerID = c.CustomerID AND c.rn = 1
ORDER BY r.Monetary DESC;
'''
df_rfm_sql = pd.read_sql_query(query_rfm, conn)
df_rfm_sql.to_csv('data/sql_exports/rfm_per_customer.csv', index=False)
print(f'[SQL 1] RFM per Customer: {len(df_rfm_sql):,} unique customers.')
display(df_rfm_sql.head())

# QUERY 2: CTE Revenue per Negara
query_country = '''
WITH country_agg AS (
    SELECT
        Country,
        COUNT(DISTINCT CustomerID) AS Total_Customers,
        COUNT(DISTINCT Invoice) AS Total_Orders,
        ROUND(SUM(Revenue), 2) AS Total_Revenue
    FROM transactions
    GROUP BY Country
)
SELECT
    Country,
    Total_Customers,
    Total_Orders,
    Total_Revenue,
    ROUND(Total_Revenue * 100.0 / SUM(Total_Revenue) OVER (), 2) AS Revenue_Share_Pct
FROM country_agg
ORDER BY Total_Revenue DESC;
'''
df_country_sql = pd.read_sql_query(query_country, conn)
df_country_sql.to_csv('data/sql_exports/revenue_by_country.csv', index=False)
print('\n[SQL 2] Top 10 Negara berdasarkan Revenue:')
display(df_country_sql.head(10))

# QUERY 3: CTE Tren Customer Baru vs Repeat per Bulan
query_monthly = '''
WITH first_purchase AS (
    SELECT
        CustomerID,
        STRFTIME('%Y-%m', MIN(InvoiceDate)) AS Cohort_Month
    FROM transactions
    GROUP BY CustomerID
),
monthly_tx AS (
    SELECT
        STRFTIME('%Y-%m', t.InvoiceDate) AS YearMonth,
        t.CustomerID,
        f.Cohort_Month,
        t.Invoice,
        t.Revenue
    FROM transactions t
    JOIN first_purchase f ON t.CustomerID = f.CustomerID
)
SELECT
    YearMonth,
    COUNT(DISTINCT CASE WHEN YearMonth = Cohort_Month THEN CustomerID END) AS New_Customers,
    COUNT(DISTINCT CASE WHEN YearMonth > Cohort_Month THEN CustomerID END) AS Repeat_Customers,
    COUNT(DISTINCT CustomerID) AS Total_Active_Customers,
    ROUND(SUM(CASE WHEN YearMonth = Cohort_Month THEN Revenue ELSE 0 END), 2) AS New_Revenue,
    ROUND(SUM(CASE WHEN YearMonth > Cohort_Month THEN Revenue ELSE 0 END), 2) AS Repeat_Revenue,
    ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM monthly_tx
GROUP BY YearMonth
ORDER BY YearMonth;
'''
df_monthly_sql = pd.read_sql_query(query_monthly, conn)
df_monthly_sql.to_csv('data/sql_exports/monthly_new_vs_repeat.csv', index=False)
print('\n[SQL 3] Tren Bulanan Customer Baru vs Repeat:')
display(df_monthly_sql.head(10))

conn.close()

## TAHAP 3 — Analisis RFM, K-Means (Elbow & Silhouette), Profil Cluster & Pareto
### 3A. RFM Scoring 1–5 & Evaluasi Jumlah Cluster (Elbow & Silhouette)

In [ ]:
rfm = df_rfm_sql.copy()

# 1. RFM Scoring 1-5
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# 2. Log-Transform + StandardScaler untuk K-Means
rfm_log = np.log1p(rfm[['Recency', 'Frequency', 'Monetary']])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_log)

# 3. Evaluasi K = 2 s.d. 8
K_range = list(range(2, 9))
inertias, silhouettes = [], []

print('Evaluasi K-Means untuk K = 2 s.d. 8:')
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    print(f'  K = {k} | Inertia = {km.inertia_:,.2f} | Silhouette = {sil:.4f}')

plt.style.use('dark_background')
fig, ax = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0F172A')
for a in ax:
    a.set_facecolor('#1E293B')
    a.spines[['top', 'right']].set_visible(False)
    a.grid(True, color='#334155', alpha=0.5, linestyle='--')

ax[0].plot(K_range, inertias, marker='o', linewidth=2.5, color='#38BDF8')
ax[0].set_title('Elbow Method (Inertia vs K)', fontsize=13, fontweight='bold', pad=12)
ax[0].set_xlabel('Jumlah Cluster (K)')
ax[0].set_ylabel('Inertia')

ax[1].plot(K_range, silhouettes, marker='s', linewidth=2.5, color='#8B5CF6')
ax[1].set_title('Silhouette Score vs K', fontsize=13, fontweight='bold', pad=12)
ax[1].set_xlabel('Jumlah Cluster (K)')
ax[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.savefig('reports/elbow_silhouette_evaluation.png', dpi=300, facecolor=fig.get_facecolor())
plt.show()

### 3B. Profil Rata-Rata Cluster, Penamaan Segmen & Analisis Pareto (Konsentrasi Revenue)

In [ ]:
OPTIMAL_K = 4

km_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm['Cluster'] = km_final.fit_predict(X_scaled)

# Urutkan Cluster 0..3 dari rata-rata Monetary tertinggi
order = rfm.groupby('Cluster')['Monetary'].mean().sort_values(ascending=False).index
rfm['Cluster'] = rfm['Cluster'].map({old: new for new, old in enumerate(order)})

# Nama segmen berdasarkan profil rata-rata asli
SEGMENT_MAP = {
    0: 'Champions',
    1: 'At Risk',
    2: 'Potential Loyalists',
    3: 'Hibernating',
}
rfm['Segment'] = rfm['Cluster'].map(SEGMENT_MAP)

cluster_profile = (
    rfm.groupby(['Cluster', 'Segment'])
    .agg(
        Customer_Count=('CustomerID', 'count'),
        Avg_Recency=('Recency', 'mean'),
        Avg_Frequency=('Frequency', 'mean'),
        Avg_Monetary=('Monetary', 'mean'),
        Total_Revenue=('Monetary', 'sum'),
        Avg_R_Score=('R_Score', 'mean'),
        Avg_F_Score=('F_Score', 'mean'),
        Avg_M_Score=('M_Score', 'mean'),
    )
    .reset_index()
)
cluster_profile['Customer_Share_Pct'] = (cluster_profile['Customer_Count'] / cluster_profile['Customer_Count'].sum() * 100).round(2)
cluster_profile['Revenue_Share_Pct'] = (cluster_profile['Total_Revenue'] / cluster_profile['Total_Revenue'].sum() * 100).round(2)
cluster_profile = cluster_profile.round(2)

print('=== PROFIL RATA-RATA TIAP CLUSTER (K = 4) ===')
display(cluster_profile)

# Analisis Pareto (Konsentrasi Revenue)
rfm_sorted = rfm.sort_values(by='Monetary', ascending=False).reset_index(drop=True)
rfm_sorted['Cum_Cust_Pct'] = (rfm_sorted.index + 1) / len(rfm_sorted) * 100
rfm_sorted['Cum_Rev_Pct'] = rfm_sorted['Monetary'].cumsum() / rfm_sorted['Monetary'].sum() * 100

pct_cust_50_rev = rfm_sorted.loc[rfm_sorted['Cum_Rev_Pct'] >= 50, 'Cum_Cust_Pct'].iloc[0]
pct_cust_80_rev = rfm_sorted.loc[rfm_sorted['Cum_Rev_Pct'] >= 80, 'Cum_Cust_Pct'].iloc[0]
n_cust_80_rev = int(rfm_sorted.loc[rfm_sorted['Cum_Rev_Pct'] >= 80].index[0] + 1)
top_20_rev = rfm_sorted.loc[rfm_sorted['Cum_Cust_Pct'] <= 20, 'Cum_Rev_Pct'].iloc[-1]

print('\n=== HASIL ANALISIS PARETO (KONSENTRASI REVENUE) ===')
print(f'- Total Unique Customers : {len(rfm_sorted):,}')
print(f'- Total Revenue Bersih   : £{rfm_sorted["Monetary"].sum():,.2f}')
print(f'- 50% Revenue disumbang oleh : {pct_cust_50_rev:.2f}% customer teratas')
print(f'- 80% Revenue disumbang oleh : {pct_cust_80_rev:.2f}% customer teratas ({n_cust_80_rev:,} customer)')
print(f'- Top 20% Customer menyumbang: {top_20_rev:.2f}% dari Total Revenue')

## TAHAP 4 — Visualisasi & Simpan PNG ke `reports/`
1. `reports/customers_per_segment.png`
2. `reports/revenue_contribution_pareto.png`
3. `reports/scatter_frequency_vs_monetary.png`
4. `reports/monthly_new_vs_repeat_trend.png`

In [ ]:
plt.style.use('dark_background')
PALETTE = ['#6366F1', '#F59E0B', '#38BDF8', '#64748B']

# 1. Chart Jumlah Customer per Segmen
fig, ax = plt.subplots(figsize=(9, 5.5), facecolor='#0F172A')
ax.set_facecolor('#1E293B')
bars = ax.bar(cluster_profile['Segment'], cluster_profile['Customer_Count'], color=PALETTE, width=0.55)
ax.set_title('Jumlah Customer per Segmen RFM (K=4)', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Jumlah Customer')
ax.set_ylim(0, cluster_profile['Customer_Count'].max() * 1.18)
ax.grid(axis='y', color='#334155', linestyle='--', alpha=0.5)
for bar, pct in zip(bars, cluster_profile['Customer_Share_Pct']):
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, yval + 40, f'{int(yval):,}\n({pct:.2f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('reports/customers_per_segment.png', dpi=300, facecolor=fig.get_facecolor())
plt.show()

# 2. Chart Kontribusi Revenue per Segmen & Kurva Kumulatif Pareto
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5), facecolor='#0F172A')
for a in ax:
    a.set_facecolor('#1E293B')
    a.spines[['top', 'right']].set_visible(False)
    a.grid(True, color='#334155', linestyle='--', alpha=0.5)

bars_rev = ax[0].bar(cluster_profile['Segment'], cluster_profile['Total_Revenue'] / 1e6, color=PALETTE, width=0.55)
ax[0].set_title('Kontribusi Revenue per Segmen (£ Juta)', fontsize=13, fontweight='bold', pad=12)
ax[0].set_ylabel('Total Revenue (£ Millions)')
ax[0].set_ylim(0, (cluster_profile['Total_Revenue'].max() / 1e6) * 1.2)
for bar, rev, pct in zip(bars_rev, cluster_profile['Total_Revenue'], cluster_profile['Revenue_Share_Pct']):
    yval = bar.get_height()
    ax[0].text(bar.get_x() + bar.get_width() / 2, yval + 0.3, f'£{rev/1e6:.2f}M\n({pct:.2f}%)', ha='center', va='bottom', fontsize=9.5, fontweight='bold')

ax[1].plot(rfm_sorted['Cum_Cust_Pct'], rfm_sorted['Cum_Rev_Pct'], color='#8B5CF6', linewidth=2.8, label='Kurva Kumulatif Revenue')
ax[1].axhline(80, color='#EF4444', linestyle='--', linewidth=1.5, label='80% Total Revenue')
ax[1].axvline(pct_cust_80_rev, color='#38BDF8', linestyle='--', linewidth=1.5, label=f'{pct_cust_80_rev:.2f}% Customer ({n_cust_80_rev:,} Cust)')
ax[1].scatter([pct_cust_80_rev], [80], color='#F59E0B', s=70, zorder=5)
ax[1].set_title('Kurva Konsentrasi Revenue (Pareto Analysis)', fontsize=13, fontweight='bold', pad=12)
ax[1].set_xlabel('% Kumulatif Customer')
ax[1].set_ylabel('% Kumulatif Revenue')
ax[1].legend(loc='lower right', frameon=True, facecolor='#0F172A', edgecolor='#334155')
plt.tight_layout()
plt.savefig('reports/revenue_contribution_pareto.png', dpi=300, facecolor=fig.get_facecolor())
plt.show()

# 3. Scatter Plot Frequency vs Monetary (Log Scale)
fig, ax = plt.subplots(figsize=(9.5, 6), facecolor='#0F172A')
ax.set_facecolor('#1E293B')
for idx, seg in enumerate(cluster_profile['Segment']):
    sub = rfm[rfm['Segment'] == seg]
    ax.scatter(sub['Frequency'], sub['Monetary'], label=f'{seg} (n={len(sub):,})', alpha=0.65, s=32, color=PALETTE[idx])
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Scatter Plot: Frequency vs Monetary per Segmen (Log Scale)', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Frequency (Jumlah Invoice Unik - Skala Log)')
ax.set_ylabel('Monetary (Total Belanja £ - Skala Log)')
ax.grid(True, which='both', color='#334155', linestyle='--', alpha=0.4)
ax.legend(title='Segmen RFM', frameon=True, facecolor='#0F172A', edgecolor='#334155')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('reports/scatter_frequency_vs_monetary.png', dpi=300, facecolor=fig.get_facecolor())
plt.show()

# 4. Line Chart Tren Customer Baru vs Repeat per Bulan
fig, ax = plt.subplots(figsize=(12.5, 5.5), facecolor='#0F172A')
ax.set_facecolor('#1E293B')
ax.plot(df_monthly_sql['YearMonth'], df_monthly_sql['New_Customers'], marker='o', linewidth=2.4, color='#38BDF8', label='New Customers')
ax.plot(df_monthly_sql['YearMonth'], df_monthly_sql['Repeat_Customers'], marker='s', linewidth=2.4, color='#8B5CF6', label='Repeat Customers')
ax.set_title('Tren Bulanan Customer Baru vs Repeat (Des 2009 – Des 2011)', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Bulan Transaksi (Year-Month)')
ax.set_ylabel('Jumlah Customer Aktif')
ax.grid(True, color='#334155', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
ax.legend(frameon=True, facecolor='#0F172A', edgecolor='#334155')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('reports/monthly_new_vs_repeat_trend.png', dpi=300, facecolor=fig.get_facecolor())
plt.show()

## TAHAP 5 — Otomatisasi Laporan Excel (`reports/rfm_segmentation.xlsx`)
Fungsi pipeline `generate_rfm_report` yang menerima path CSV bersih dan menghasilkan file Excel multi-sheet (`RFM_Data`, `Cluster_Profile`, `Monthly`).

In [ ]:
def generate_rfm_report(
    csv_path: str,
    output_excel_path: str = 'reports/rfm_segmentation.xlsx',
    n_clusters: int = 4,
    segment_map: dict = None,
):
    df_in = pd.read_csv(csv_path)
    df_in['InvoiceDate'] = pd.to_datetime(df_in['InvoiceDate'])
    df_in['CustomerID'] = df_in['CustomerID'].astype(str)

    snapshot_date = df_in['InvoiceDate'].max() + pd.Timedelta(days=1)

    cust_country = (
        df_in.groupby(['CustomerID', 'Country'])
        .size()
        .reset_index(name='tx_count')
        .sort_values(['CustomerID', 'tx_count'], ascending=[True, False])
        .drop_duplicates('CustomerID')[['CustomerID', 'Country']]
    )

    rfm_df = (
        df_in.groupby('CustomerID')
        .agg(
            Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
            Frequency=('Invoice', 'nunique'),
            Monetary=('Revenue', lambda x: round(x.sum(), 2)),
        )
        .reset_index()
    )
    rfm_df = rfm_df.merge(cust_country, on='CustomerID', how='left')

    rfm_df['R_Score'] = pd.qcut(rfm_df['Recency'].rank(method='first'), q=5, labels=[5, 4, 3, 2, 1]).astype(int)
    rfm_df['F_Score'] = pd.qcut(rfm_df['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    rfm_df['M_Score'] = pd.qcut(rfm_df['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(str) + rfm_df['F_Score'].astype(str) + rfm_df['M_Score'].astype(str)

    X_log = np.log1p(rfm_df[['Recency', 'Frequency', 'Monetary']])
    X_std = StandardScaler().fit_transform(X_log)
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    rfm_df['Cluster'] = km.fit_predict(X_std)

    order = rfm_df.groupby('Cluster')['Monetary'].mean().sort_values(ascending=False).index
    rfm_df['Cluster'] = rfm_df['Cluster'].map({old: new for new, old in enumerate(order)})

    if segment_map is None:
        segment_map = {
            0: 'Champions',
            1: 'At Risk',
            2: 'Potential Loyalists',
            3: 'Hibernating',
        }
    rfm_df['Segment'] = rfm_df['Cluster'].map(segment_map)

    rfm_data_sheet = (
        rfm_df[['CustomerID', 'Country', 'Recency', 'Frequency', 'Monetary', 'RFM_Score', 'Segment', 'R_Score', 'F_Score', 'M_Score']]
        .sort_values(by='Monetary', ascending=False)
        .reset_index(drop=True)
    )

    profile_sheet = (
        rfm_df.groupby(['Cluster', 'Segment'])
        .agg(
            Customer_Count=('CustomerID', 'count'),
            Avg_Recency=('Recency', 'mean'),
            Avg_Frequency=('Frequency', 'mean'),
            Avg_Monetary=('Monetary', 'mean'),
            Total_Revenue=('Monetary', 'sum'),
            Avg_R_Score=('R_Score', 'mean'),
            Avg_F_Score=('F_Score', 'mean'),
            Avg_M_Score=('M_Score', 'mean'),
        )
        .reset_index()
    )
    profile_sheet['Customer_Share_Pct'] = (profile_sheet['Customer_Count'] / profile_sheet['Customer_Count'].sum() * 100).round(2)
    profile_sheet['Revenue_Share_Pct'] = (profile_sheet['Total_Revenue'] / profile_sheet['Total_Revenue'].sum() * 100).round(2)
    profile_sheet = profile_sheet.round(2)

    df_in['YearMonth'] = df_in['InvoiceDate'].dt.strftime('%Y-%m')
    cohort = df_in.groupby('CustomerID')['YearMonth'].min().reset_index().rename(columns={'YearMonth': 'Cohort_Month'})
    df_m = df_in.merge(cohort, on='CustomerID', how='left')
    df_m['Customer_Type'] = np.where(df_m['YearMonth'] == df_m['Cohort_Month'], 'New', 'Repeat')

    monthly_sheet = (
        df_m.groupby('YearMonth')
        .agg(
            New_Customers=('CustomerID', lambda x: x[df_m.loc[x.index, 'Customer_Type'] == 'New'].nunique()),
            Repeat_Customers=('CustomerID', lambda x: x[df_m.loc[x.index, 'Customer_Type'] == 'Repeat'].nunique()),
            Total_Active_Customers=('CustomerID', 'nunique'),
            New_Revenue=('Revenue', lambda x: round(x[df_m.loc[x.index, 'Customer_Type'] == 'New'].sum(), 2)),
            Repeat_Revenue=('Revenue', lambda x: round(x[df_m.loc[x.index, 'Customer_Type'] == 'Repeat'].sum(), 2)),
            Total_Revenue=('Revenue', lambda x: round(x.sum(), 2)),
        )
        .reset_index()
        .sort_values('YearMonth')
    )

    os.makedirs(os.path.dirname(output_excel_path), exist_ok=True)
    with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
        rfm_data_sheet.to_excel(writer, sheet_name='RFM_Data', index=False)
        profile_sheet.to_excel(writer, sheet_name='Cluster_Profile', index=False)
        monthly_sheet.to_excel(writer, sheet_name='Monthly', index=False)

    print(f'Laporan Excel berhasil dibuat di: {output_excel_path}')
    return rfm_data_sheet, profile_sheet, monthly_sheet


rfm_sheet, prof_sheet, mon_sheet = generate_rfm_report(
    csv_path='data/clean/retail_clean.csv',
    output_excel_path='reports/rfm_segmentation.xlsx',
    n_clusters=OPTIMAL_K,
    segment_map=SEGMENT_MAP,
)